# Step 5 live LLM run on Google Colab

Runs Step 5.1 (a local 4-bit Qwen 2.5 7B Instruct model) and Step 5.6 (the
canonical Maryland → Boston check) on a free Colab GPU, then zips the outputs.

1. **Runtime → Change runtime type → T4 GPU** (any GPU works), then **Save**.
2. **Runtime → Run all.** It takes about 20–40 minutes; the model download is ~15 GB.
3. When the last cell finishes, your browser downloads `step5_colab_out.zip`.
   Send that file back (put it in the repo folder or share it).

If a cell fails, copy its error output and send it instead.

In [ ]:
# 1. Confirm a GPU is attached (should list a GPU, e.g. Tesla T4).
!nvidia-smi

In [ ]:
# 2. Get the code (fixed Step 5/6 branch) and install it with the LLM extras.
!rm -rf /content/ShouldAgentsVerifyOrRepair
!git clone -q -b step5-6-llm-costs https://github.com/shohinirheasarkar/ShouldAgentsVerifyOrRepair.git /content/ShouldAgentsVerifyOrRepair
%cd /content/ShouldAgentsVerifyOrRepair
!git log -1 --format='code commit: %H'
!pip install -q '.[dev,llm]'

In [ ]:
# 3. Core offline test suite (no GPU needed; should end with "passed").
!python -m pytest -q

In [ ]:
# 4. Download the model weights once (online).
import os
os.environ["HF_HOME"] = "/content/hf_cache"
!python scripts/prefetch_llm_model.py --model Qwen/Qwen2.5-7B-Instruct --output-json /content/hf_cache/qwen25_7b_snapshot.json

In [ ]:
# 5. Everything after this point runs offline from the downloaded weights.
import json
os.environ.update({
    "HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1", "TOKENIZERS_PARALLELISM": "false",
    "PYTHONPATH": "/content/ShouldAgentsVerifyOrRepair/src",
})
REVISION = json.load(open("/content/hf_cache/qwen25_7b_snapshot.json"))["model_revision"]
os.environ["LLM_REVISION"] = REVISION
OUT = "/content/step5_colab_out"
!rm -rf {OUT}
print("model revision:", REVISION)

In [ ]:
# 6. Live 4-bit integration test (Step 5.1).
!RUN_LLM=1 python -m pytest -o addopts='' tests/test_llm_optional.py -v 2>&1 | tee /content/live_test.log

In [ ]:
# 7. Canonical Maryland -> Boston check (Step 5.6) and the toy policy runs, one model load.
#    T4 GPUs are slower than Zaratan's H100s, so the per-call timeout is raised to 180 s.
!python scripts/run_step5_artifacts.py --mode llm --llm-model Qwen/Qwen2.5-7B-Instruct \
    --llm-revision {REVISION} --llm-quantization nf4 --llm-dtype auto \
    --llm-timeout 180 --llm-max-retries 2 --output-dir {OUT} 2>&1 | tee /content/artifacts.log

In [ ]:
# 8. Standalone CLI toy command (Step 5.1 smoke check).
!python -m verify_repair.cli run-toy --policy verify-only --mode llm \
    --llm-model Qwen/Qwen2.5-7B-Instruct --llm-revision {REVISION} \
    --llm-quantization nf4 --llm-dtype auto --llm-timeout 180 \
    --answers-jsonl {OUT}/cli_smoke_verify_only_answers.jsonl 2>&1 | tee /content/cli_smoke.log

In [ ]:
# 9. Save environment details and logs, zip everything, and download it.
!mkdir -p {OUT}
!nvidia-smi > {OUT}/nvidia_smi.txt
!python --version > {OUT}/environment.txt 2>&1
!pip list 2>/dev/null | grep -iE "^(torch|transformers|bitsandbytes|accelerate) " >> {OUT}/environment.txt
!git -C /content/ShouldAgentsVerifyOrRepair log -1 --format='code commit: %H' >> {OUT}/environment.txt
!cp /content/live_test.log /content/artifacts.log /content/cli_smoke.log {OUT}/ 2>/dev/null
!cd /content && zip -qr step5_colab_out.zip step5_colab_out
from google.colab import files
files.download("/content/step5_colab_out.zip")